[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/laboratorios/S13/S13_02_plantilla_informe_visual.ipynb)

# S13 · 02 — Plantilla: informe visual de una página para dirección

**Tiempo estimado:** 1 h (parte del reto PBL "Informe visual para dirección") · **Módulo:** M5

Esta es una **plantilla que ya funciona**: al ejecutarla de arriba abajo genera un panel de una página con
el modelo de cancelaciones de TurisData. Tu trabajo en el reto es **adaptarla a tu mejor modelo del curso**
y escribir el mensaje para dirección. Busca las celdas marcadas con `# >>> EDITA` .

**Estructura del informe (una página):**
1. **Titular:** la decisión que recomiendas, en una frase.
2. **Cuatro gráficos** (2 × 2): fiabilidad del modelo, qué mueve el resultado, cuánto vale, y un dato de negocio.
3. **Tres viñetas:** qué hemos hecho, qué recomendamos, qué límites tiene.
4. **Pie:** datos, fecha, modelo y responsable.

Entrega: el notebook + el PNG generado + el informe en Markdown (`informe_visual.md`, plantilla en la guía).

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score,
                             precision_recall_curve, average_precision_score)
from sklearn.inspection import permutation_importance

SEMILLA = 42
np.random.seed(SEMILLA)
sns.set_theme(style="whitegrid")

## Parámetros del informe (edítalos)

In [ ]:
# >>> EDITA: tu mensaje y tus costes
TITULAR = "Recomendamos pedir depósito y avisar a las reservas de riesgo alto"
CLIENTE = "TurisData Canarias · Dirección de Ingresos"
AUTORES = "Equipo X (nombres)"
COSTE_FN = 50   # € que perdemos por cada cancelación que el modelo no detecta
COSTE_FP = 30   # € que cuesta tratar como "en riesgo" a quien sí viene
UMBRAL = 0.35   # a partir de qué probabilidad avisamos (elígelo con el gráfico de costes del laboratorio 01)

## 1. Datos y modelo
>>> EDITA: sustituye el modelo por **tu mejor modelo del curso** (mismo formato: un objeto con `predict_proba`
que reciba `X_test`). Mantén las **semillas** y la separación entrenamiento/prueba.

In [ ]:
reservas = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_reserva", "fecha_llegada"])
reservas["mes_llegada"] = reservas["fecha_llegada"].dt.month
X = reservas.drop(columns=["id_reserva", "fecha_reserva", "fecha_llegada", "cancelada"])
y = reservas["cancelada"]
X_ent, X_test, y_ent, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEMILLA)

categoricas = X.select_dtypes("object").columns.tolist()
numericas = [c for c in X.columns if c not in categoricas]
prep = ColumnTransformer([("num", StandardScaler(), numericas), ("cat", OneHotEncoder(handle_unknown="ignore"), categoricas)])
modelo = Pipeline([("prep", prep), ("m", LogisticRegression(max_iter=2000))]).fit(X_ent, y_ent)   # >>> EDITA
prob = modelo.predict_proba(X_test)[:, 1]
print("AUC en prueba:", round(roc_auc_score(y_test, prob), 3), "· reservas de prueba:", len(y_test))

## 2. Cálculos que alimentan el panel

In [ ]:
pred = (prob >= UMBRAL).astype(int)
cm = confusion_matrix(y_test, pred, normalize="true")
fpr, tpr, _ = roc_curve(y_test, prob)
auc = roc_auc_score(y_test, prob)
res = permutation_importance(modelo, X_test, y_test, scoring="roc_auc", n_repeats=5, random_state=SEMILLA)
importancias = pd.Series(res.importances_mean, index=X_test.columns).sort_values(ascending=False)

fn = int(((pred == 0) & (y_test == 1)).sum())
fp = int(((pred == 1) & (y_test == 0)).sum())
coste_con_modelo = COSTE_FN * fn + COSTE_FP * fp
coste_sin_modelo = COSTE_FN * int((y_test == 1).sum())   # sin modelo: no detectamos ninguna cancelación
ahorro = coste_sin_modelo - coste_con_modelo
print(f"Coste sin modelo: {coste_sin_modelo:,.0f} € · con modelo: {coste_con_modelo:,.0f} € · ahorro: {ahorro:,.0f} € (en {len(y_test)} reservas de prueba)")

## 3. El panel de una página
Reglas: título del panel = decisión; título de cada gráfico = conclusión; ejes con nombre; pocos colores.

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle(TITULAR, fontsize=16, fontweight="bold", x=0.02, ha="left")

# (1) Fiabilidad: matriz de confusión
ConfusionMatrixDisplay(cm, display_labels=["No cancela", "Cancela"]).plot(ax=ax[0, 0], cmap="Blues", values_format=".0%", colorbar=False)
ax[0, 0].set_title(f"Detectamos el {cm[1, 1]:.0%} de las cancelaciones,\ncon un {cm[0, 1]:.0%} de falsas alarmas")   # >>> EDITA el título si cambian los números
ax[0, 0].set_xlabel("Lo que dijo el modelo"); ax[0, 0].set_ylabel("Lo que ocurrió")

# (2) Fiabilidad: curva ROC
ax[0, 1].plot(fpr, tpr, color="tab:blue", label=f"Modelo (AUC = {auc:.2f})")
ax[0, 1].plot([0, 1], [0, 1], "--", color="gray", label="Azar")
ax[0, 1].set_title("Acierta bastante mejor que el azar, pero no es perfecto")   # >>> EDITA
ax[0, 1].set_xlabel("Falsas alarmas"); ax[0, 1].set_ylabel("Cancelaciones detectadas"); ax[0, 1].legend(loc="lower right")

# (3) Qué mueve el resultado
importancias.head(6)[::-1].plot.barh(ax=ax[1, 0], color="tab:blue")
ax[1, 0].set_title("Lo que más pesa en una cancelación")   # >>> EDITA: nombra la variable líder
ax[1, 0].set_xlabel("Caída del AUC al desordenar la variable")

# (4) Valor para el negocio
valores = [coste_sin_modelo, coste_con_modelo]
barras = ax[1, 1].bar(["Sin modelo", "Con modelo"], valores, color=["lightgray", "tab:green"])
ax[1, 1].bar_label(barras, fmt="{:,.0f} €", padding=3)
ax[1, 1].set_title(f"Con el modelo perdemos {ahorro:,.0f} € menos en {len(y_test)} reservas")   # >>> EDITA
ax[1, 1].set_ylabel("Coste de las cancelaciones (€)"); ax[1, 1].set_ylim(0, max(valores) * 1.15)

fig.text(0.02, 0.005, f"{CLIENTE} · {AUTORES} · Datos sintéticos de TurisData · modelo: regresión logística · umbral {UMBRAL:.2f} · semilla {SEMILLA}",
         fontsize=8, color="gray")
sns.despine()
plt.tight_layout(rect=(0, 0.02, 1, 0.96))
plt.savefig("informe_visual_panel.png", dpi=150)
plt.show()

## 4. El texto del informe (tres viñetas)
>>> EDITA: sustituye lo que está entre corchetes. Escribe para una persona de dirección que **no** sabe
Machine Learning: frases cortas, cifras concretas, sin jerga.

- **Qué hemos hecho:** [Un modelo que estima la probabilidad de cancelación de cada reserva usando X reservas anteriores.]
- **Qué recomendamos:** [Acción concreta, a quién, con qué umbral y qué ahorro esperado.]
- **Qué límites tiene:** [Datos sintéticos / periodo cubierto / falla más con…; qué NO debe usarse para decidir; cómo se revisará.]

**Checklist antes de entregar**
- [ ] El titular es una decisión, no un resultado.
- [ ] Cada gráfico tiene un título-conclusión y ejes con nombre.
- [ ] Las cifras del texto coinciden con las del gráfico.
- [ ] Se dice con cuántos datos y con qué modelo.
- [ ] Hay al menos un límite y un riesgo (sesgo, uso indebido) mencionados.
- [ ] El notebook se ejecuta de arriba abajo (reinicia y ejecuta todo).